# CycleGAN: Photo ↔ Monet

Notebook ini melatih dua generator untuk menerjemahkan foto menjadi gaya Monet dan sebaliknya, tanpa pasangan gambar yang cocok. Perubahan utama: cycle consistency dan identity loss, checkpoint yang bisa dilanjutkan, serta evaluasi visual pada data validasi yang tidak dipakai training.

Training penuh tidak dijalankan otomatis. Mulai dari atas, periksa konfigurasi, lalu jalankan cell training.

In [1]:
import random
import sys
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

EPOCHS = 10                 # Target total epoch; aman dilanjutkan dari checkpoint
BATCH_SIZE = 1
LAMBDA_CYCLE = 10.0
LAMBDA_IDENTITY = 5.0       # 0.5 * LAMBDA_CYCLE
NUM_WORKERS = 0


Device: cuda
GPU: NVIDIA GeForce RTX 4060


/mnt/media/Projects/Personal/App/GAN_Art/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Simpan checkpoint di Google Drive (Colab)

File di `/content` ada di VM sementara Colab dan bisa hilang saat runtime dihapus. Cell berikut memasang Drive; beri otorisasi saat diminta. Hanya checkpoint dan gambar evaluasi yang disimpan di Drive, dataset tetap dibaca dari disk runtime.

In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
    CHECKPOINT_ROOT = Path("/content/drive/MyDrive/GAN_Art/checkpoints/cyclegan_v2")
    old_checkpoint = Path("/content/checkpoints/cyclegan_v2/latest.pt")
    drive_checkpoint = CHECKPOINT_ROOT / "latest.pt"
    if old_checkpoint.exists() and not drive_checkpoint.exists():
        CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
        shutil.copy2(old_checkpoint, drive_checkpoint)
        print("Checkpoint lokal lama disalin ke Drive:", drive_checkpoint)
except ModuleNotFoundError:
    IN_COLAB = False
    CHECKPOINT_ROOT = None


## Data dan split validasi

10% gambar tiap domain ditahan untuk validasi. Foto dan Monet tidak dipasangkan; keduanya hanya perlu berasal dari domain yang tepat. Resize menjaga rasio aspek, lalu crop menghasilkan input 256×256. Validasi memakai center crop yang deterministik.

In [3]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
DATASET_PATH = PROJECT_ROOT / "dataset" / "gan-getting-started"
CHECKPOINT_DIR = CHECKPOINT_ROOT if IN_COLAB else PROJECT_ROOT / "checkpoints" / "cyclegan_v2"
SAMPLE_DIR = CHECKPOINT_DIR / "samples"

MONET_FILES = sorted((DATASET_PATH / "monet_jpg").glob("*.jpg"))
PHOTO_FILES = sorted((DATASET_PATH / "photo_jpg").glob("*.jpg"))
if not MONET_FILES or not PHOTO_FILES:
    raise FileNotFoundError(f"Dataset tidak ditemukan atau kosong: {DATASET_PATH}")

def split_files(files, val_fraction=0.1):
    files = list(files)
    random.Random(SEED).shuffle(files)
    n_val = max(1, round(len(files) * val_fraction))
    return files[n_val:], files[:n_val]

monet_train_files, monet_val_files = split_files(MONET_FILES)
photo_train_files, photo_val_files = split_files(PHOTO_FILES)

train_transform = transforms.Compose([
    transforms.Resize(286, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomCrop(256),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3),
])
val_transform = transforms.Compose([
    transforms.Resize(286, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3),
])

class ImageDataset(Dataset):
    def __init__(self, filenames, transform):
        self.filenames = filenames
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, index):
        with Image.open(self.filenames[index]) as image:
            return self.transform(image.convert("RGB"))

monet_train_ds = ImageDataset(monet_train_files, train_transform)
photo_train_ds = ImageDataset(photo_train_files, train_transform)
monet_val_ds = ImageDataset(monet_val_files, val_transform)
photo_val_ds = ImageDataset(photo_val_files, val_transform)

monet_loader = DataLoader(monet_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
photo_loader = DataLoader(photo_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
print(f"Train: {len(photo_train_ds)} photo, {len(monet_train_ds)} Monet")
print(f"Validasi: {len(photo_val_ds)} photo, {len(monet_val_ds)} Monet")
print("Checkpoint dir:", CHECKPOINT_DIR)


Train: 6334 photo, 270 Monet
Validasi: 704 photo, 30 Monet
Checkpoint dir: /mnt/media/Projects/Personal/App/GAN_Art/checkpoints/cyclegan_v2


## Model

Generator memakai dua tahap downsampling, residual blocks, dan upsampling. Discriminator menilai patch lokal gambar (PatchGAN).

In [4]:
from gan_art.models import Discriminator, Generator

In [5]:
photo_to_monet = Generator().to(device)
monet_to_photo = Generator().to(device)
monet_discriminator = Discriminator().to(device)
photo_discriminator = Discriminator().to(device)

criterion_gan = nn.MSELoss()  # Least-squares GAN objective
criterion_l1 = nn.L1Loss()
optimizer_G = torch.optim.Adam(
    list(photo_to_monet.parameters()) + list(monet_to_photo.parameters()),
    lr=2e-4, betas=(0.5, 0.999),
)
optimizer_D = torch.optim.Adam(
    list(monet_discriminator.parameters()) + list(photo_discriminator.parameters()),
    lr=2e-4, betas=(0.5, 0.999),
)


## Evaluasi tetap

Gunakan gambar validasi yang sama setiap kali membandingkan checkpoint. Grid menampilkan input, hasil terjemahan, lalu hasil terjemahan balik. Nilai `cycle L1` mengukur seberapa dekat rekonstruksi ke input setelah dua arah terjemahan; nilainya membantu memantau konsistensi, tetapi **bukan skor kualitas gaya**.

In [6]:
@torch.inference_mode()
def evaluate(n=4, save_path=None, show=True):
    n = min(n, len(photo_val_ds), len(monet_val_ds))
    if n < 1:
        raise ValueError("Validasi memerlukan minimal satu gambar di tiap domain")

    models = (photo_to_monet, monet_to_photo)
    was_training = [model.training for model in models]
    for model in models:
        model.eval()

    photo_cycle_errors, monet_cycle_errors = [], []
    fig, axes = plt.subplots(6, n, figsize=(3 * n, 15), squeeze=False)
    row_titles = [
        "Photo input", "Photo → Monet", "Rekonstruksi photo",
        "Monet input", "Monet → photo", "Rekonstruksi Monet",
    ]
    for row, title in enumerate(row_titles):
        axes[row, 0].set_ylabel(title, fontsize=10)

    for i in range(n):
        photo = photo_val_ds[i].unsqueeze(0).to(device)
        monet = monet_val_ds[i].unsqueeze(0).to(device)
        fake_monet = photo_to_monet(photo)
        rec_photo = monet_to_photo(fake_monet)
        fake_photo = monet_to_photo(monet)
        rec_monet = photo_to_monet(fake_photo)
        photo_cycle_errors.append(F.l1_loss(rec_photo, photo).item())
        monet_cycle_errors.append(F.l1_loss(rec_monet, monet).item())

        images = (photo, fake_monet, rec_photo, monet, fake_photo, rec_monet)
        for row, image in enumerate(images):
            pixels = ((image[0].cpu().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).numpy()
            axes[row, i].imshow(pixels)
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_titles[row], loc="left", fontsize=9)

    metrics = {
        "photo_cycle_l1": float(np.mean(photo_cycle_errors)),
        "monet_cycle_l1": float(np.mean(monet_cycle_errors)),
    }
    fig.suptitle(
        f"Validation | Photo cycle L1: {metrics['photo_cycle_l1']:.4f} | "
        f"Monet cycle L1: {metrics['monet_cycle_l1']:.4f}"
    )
    fig.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=130, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)

    for model, training in zip(models, was_training):
        model.train(training)
    return metrics


## Training dan checkpoint

Checkpoint `latest.pt` disimpan setelah setiap epoch berisi bobot keempat model, optimizer, epoch, dan riwayat loss/evaluasi. Di Colab, checkpoint dan grid evaluasi masuk ke Google Drive; bila runtime berhenti di tengah epoch, file terakhir berisi epoch penuh sebelumnya. Setelah membuka runtime baru, jalankan notebook sampai model terbentuk, set `RESUME = True`, lalu jalankan training lagi. `EPOCHS` adalah target total epoch—misalnya checkpoint epoch 5 dengan `EPOCHS = 10` akan melanjutkan dari epoch 6.

Untuk memakai model tanpa melanjutkan training, panggil `load_generators_for_inference()` setelah model dibuat. Checkpoint lama dari arsitektur sebelum `cyclegan_v2` tidak kompatibel.

In [7]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / "latest.pt"


def save_checkpoint(epoch, history):
    state = {
        "epoch": epoch,
        "photo_to_monet": photo_to_monet.state_dict(),
        "monet_to_photo": monet_to_photo.state_dict(),
        "monet_discriminator": monet_discriminator.state_dict(),
        "photo_discriminator": photo_discriminator.state_dict(),
        "optimizer_G": optimizer_G.state_dict(),
        "optimizer_D": optimizer_D.state_dict(),
        "history": history,
    }
    temporary_path = CHECKPOINT_PATH.with_suffix(".tmp")
    torch.save(state, temporary_path)
    temporary_path.replace(CHECKPOINT_PATH)


def load_checkpoint(path=CHECKPOINT_PATH):
    state = torch.load(path, map_location=device, weights_only=False)
    photo_to_monet.load_state_dict(state["photo_to_monet"])
    monet_to_photo.load_state_dict(state["monet_to_photo"])
    monet_discriminator.load_state_dict(state["monet_discriminator"])
    photo_discriminator.load_state_dict(state["photo_discriminator"])
    optimizer_G.load_state_dict(state["optimizer_G"])
    optimizer_D.load_state_dict(state["optimizer_D"])
    return state["epoch"], state["history"]


def load_generators_for_inference(path=CHECKPOINT_PATH):
    state = torch.load(path, map_location=device, weights_only=False)
    photo_to_monet.load_state_dict(state["photo_to_monet"])
    monet_to_photo.load_state_dict(state["monet_to_photo"])
    photo_to_monet.eval()
    monet_to_photo.eval()
    return state["epoch"]


def repeat_loader(loader):
    while True:
        yield from loader


def train(epochs, start_epoch=0, history=None):
    history = list(history or [])
    steps_per_epoch = max(len(photo_loader), len(monet_loader))
    photo_batches = iter(repeat_loader(photo_loader))
    monet_batches = iter(repeat_loader(monet_loader))

    epoch_bar = tqdm(range(start_epoch, epochs), desc="Training", unit="epoch")
    for epoch in epoch_bar:
        totals = torch.zeros(4, device=device)
        batch_bar = tqdm(
            range(steps_per_epoch),
            desc=f"Epoch {epoch + 1}/{epochs}",
            unit="batch",
            leave=False,
        )
        for step in batch_bar:
            batch_bar.set_postfix_str("stage: load image pair")
            real_photo = next(photo_batches).to(device)
            real_monet = next(monet_batches).to(device)

            batch_bar.set_postfix_str("stage: translation + adversarial scores")
            optimizer_G.zero_grad(set_to_none=True)
            fake_monet = photo_to_monet(real_photo)
            fake_photo = monet_to_photo(real_monet)
            pred_fake_monet_for_g = monet_discriminator(fake_monet)
            pred_fake_photo_for_g = photo_discriminator(fake_photo)
            loss_adv = (
                criterion_gan(pred_fake_monet_for_g, torch.ones_like(pred_fake_monet_for_g))
                + criterion_gan(pred_fake_photo_for_g, torch.ones_like(pred_fake_photo_for_g))
            )
            batch_bar.set_postfix_str("stage: cycle reconstruction")
            rec_photo = monet_to_photo(fake_monet)
            rec_monet = photo_to_monet(fake_photo)
            loss_cycle = criterion_l1(rec_photo, real_photo) + criterion_l1(rec_monet, real_monet)
            batch_bar.set_postfix_str("stage: identity preservation")
            identity_monet = photo_to_monet(real_monet)
            identity_photo = monet_to_photo(real_photo)
            loss_identity = criterion_l1(identity_monet, real_monet) + criterion_l1(identity_photo, real_photo)
            loss_G = loss_adv + LAMBDA_CYCLE * loss_cycle + LAMBDA_IDENTITY * loss_identity
            batch_bar.set_postfix_str("stage: generator backward + update")
            loss_G.backward()
            optimizer_G.step()

            batch_bar.set_postfix_str("stage: discriminator forward + loss")
            optimizer_D.zero_grad(set_to_none=True)
            pred_real_monet = monet_discriminator(real_monet)
            pred_fake_monet = monet_discriminator(fake_monet.detach())
            loss_D_monet = (
                criterion_gan(pred_real_monet, torch.ones_like(pred_real_monet))
                + criterion_gan(pred_fake_monet, torch.zeros_like(pred_fake_monet))
            )
            pred_real_photo = photo_discriminator(real_photo)
            pred_fake_photo = photo_discriminator(fake_photo.detach())
            loss_D_photo = (
                criterion_gan(pred_real_photo, torch.ones_like(pred_real_photo))
                + criterion_gan(pred_fake_photo, torch.zeros_like(pred_fake_photo))
            )
            loss_D = loss_D_monet + loss_D_photo
            batch_bar.set_postfix_str("stage: discriminator backward + update")
            loss_D.backward()
            optimizer_D.step()

            totals += torch.stack((loss_G.detach(), loss_D.detach(), loss_cycle.detach(), loss_identity.detach()))
            if (step + 1) % 100 == 0 or step + 1 == steps_per_epoch:
                batch_bar.set_postfix(
                    stage="batch complete",
                    G=f"{loss_G.detach().item():.3f}",
                    D=f"{loss_D.detach().item():.3f}",
                    cycle=f"{loss_cycle.detach().item():.3f}",
                )

        mean_G, mean_D, mean_cycle, mean_identity = (totals / steps_per_epoch).cpu().tolist()
        epoch_bar.set_postfix(stage="validation")
        metrics = evaluate(n=4, save_path=SAMPLE_DIR / f"epoch_{epoch + 1:03d}.png", show=False)
        history.append({
            "epoch": epoch + 1,
            "loss_G": mean_G,
            "loss_D": mean_D,
            "loss_cycle": mean_cycle,
            "loss_identity": mean_identity,
            **metrics,
        })
        epoch_bar.set_postfix(stage="saving checkpoint")
        save_checkpoint(epoch + 1, history)
        epoch_bar.set_postfix(
            stage="epoch complete",
            G=f"{mean_G:.3f}",
            D=f"{mean_D:.3f}",
            val_cycle=f"{metrics['photo_cycle_l1']:.3f}/{metrics['monet_cycle_l1']:.3f}",
        )
        print(
            f"Epoch [{epoch + 1}/{epochs}] "
            f"G: {mean_G:.4f} | D: {mean_D:.4f} | "
            f"cycle: {mean_cycle:.4f} | identity: {mean_identity:.4f} | "
            f"val cycle L1 photo/Monet: {metrics['photo_cycle_l1']:.4f}/"
            f"{metrics['monet_cycle_l1']:.4f}"
        )
    return history


In [8]:
RESUME = True
START_EPOCH, history = 0, []
if RESUME and CHECKPOINT_PATH.exists():
    START_EPOCH, history = load_checkpoint()
    print(f"Melanjutkan dari epoch {START_EPOCH}")

history = train(EPOCHS, start_epoch=START_EPOCH, history=history)


Melanjutkan dari epoch 4


Training:   0%|          | 0/6 [00:29<?, ?epoch/s]


KeyboardInterrupt: 

## Membaca hasil evaluasi

Hasil yang baik perlu memenuhi semuanya, bukan sekadar loss turun:

- **Isi tetap terjaga:** objek, bentuk, garis, dan komposisi foto masih dikenali pada baris `Photo → Monet`; begitu juga sebaliknya.
- **Gaya tujuan terlihat:** hasil foto-ke-Monet menunjukkan tekstur/warna sapuan ala Monet, tetapi tidak menjadi kabur, penuh artefak, atau sekadar diberi tint warna.
- **Konsisten pada gambar yang tidak dilatih:** periksa beberapa input validasi. Hasil tidak runtuh menjadi gambar yang hampir sama untuk semua input (*mode collapse*).
- **Rekonstruksi masuk akal:** baris rekonstruksi seharusnya mendekati gambar input. Cycle L1 yang turun mendukung konsistensi ini, tetapi tidak membuktikan bahwa gaya terjemahannya bagus.

Tidak ada target gambar pasangan yang tepat untuk validasi unpaired ini. Karena itu jangan memilih model dari satu angka loss saja: bandingkan grid validasi epoch-ke-epoch, lalu lihat beberapa gambar lain yang tidak dilatih. Grid tiap epoch tersimpan di `checkpoints/cyclegan_v2/samples/`; bobot terbaru ada di `checkpoints/cyclegan_v2/latest.pt`.

In [ ]:
validation_metrics = evaluate(n=4, save_path=SAMPLE_DIR / "validation_latest.png", show=True)
print(validation_metrics)
